In [3]:
import torch
import torch.nn.functional as F

import pandas as pd

import preprocess
import update_model
import validate

In [19]:
def get_device():
    if torch.backends.mps.is_available():
        print("MPS is available")
        return torch.device("mps")
    elif torch.cuda.is_available():
        print("CUDA is available")
        return torch.device("cuda")
    else:
        print("CPU is available")
        return torch.device("cpu")

DEVICE = get_device()
print('Using device:', str(DEVICE).upper(), "\n")

MODEL_DIR="./model"
DATA_PATH = "war_and_peace.txt"

TRAIN_DATA_BUCKET = "cbow-training-data-1f656"
MODEL_DATA_BUCKET = "cbow-model-data-1f656"

MPS is available
Using device: MPS 



In [ ]:
# # === Run this code for the first model initialization ===

# # Download training data
# preprocess.download_data_from_gcs(
#     TRAIN_DATA_BUCKET,
#     DATA_PATH,
# )

# model, cfg = update_model.init_first_model(data_path=DATA_PATH)

In [20]:
# === Run this code to initialize pretrained model ===

# Download training data
preprocess.download_data_from_gcs(
    TRAIN_DATA_BUCKET,
    DATA_PATH
)

# Download pretrained model and its configuration
preprocess.download_model_from_gcs(
    bucket_name=MODEL_DATA_BUCKET,
    download_dir=MODEL_DIR,
)


 -- File has been downloaded in 'data/war_and_peace.txt'
Latest version folder: version-3
Model already exists at ./model/model_cpu.pt
Model already exists at ./model/model_ns_config.json
Model already exists at ./model/sequence.pt


In [21]:
# === Initialize pretrained model ===
model, cfg = update_model.init_model(
    new_data_path=f"data/{DATA_PATH}"
)

 CBOW Incremental Update
   new data : data/war_and_peace.txt
   device   : MPS
 -- Loaded weights from ./model/model_cpu.pt
 -- Existing vocab size : 18,526
 -- New words added     : 31,100
 -- Merged vocab size   : 49,626
 -- Sequence length: 8,122,611 tokens  (saved → ./model/sequence.pt)
 -- Resizing model: 18,526 → 49,626 vocab entries
 -- Old embeddings size: torch.Size([18526, 256])
 -- New embeddings size:torch.Size([49626, 256])
 -- Old embeddings size: torch.Size([18526, 256])
 -- New embeddings size:torch.Size([49626, 256])

 -- Done ✓


In [31]:
# === Train model ===
model = update_model.train(
    model=model,
    cfg=cfg,
    device=DEVICE,
    epochs=5,
)

# === Save model configuration locally ===
update_model.save(model, cfg)


 -- Training for 5 epoch(s) on 8,122,607 samples (NEG k=5, device=MPS)

   Epoch   1/5  loss=2734.0069  time=154.7s
   Epoch   2/5  loss=2718.9328  time=151.9s
   Epoch   3/5  loss=2715.5375  time=152.1s
   Epoch   4/5  loss=2713.4884  time=150.3s
   Epoch   5/5  loss=2713.7150  time=151.1s

 -- Weights  saved → ./model/model_cpu.pt
 -- Config   saved → ./model/model_ns_config.json


In [10]:
# # === Save model configuration in GCS and create new version ===
# preprocess.save_model_to_gcs(bucket_name=MODEL_DATA_BUCKET)

In [32]:
model = model.to(DEVICE)

word_embeddings = model.get_input_embeddings() # Shape: (vocab_size, embedding_dim)
word_embeddings_n = F.normalize(word_embeddings, p=2, dim=1)

window_size = cfg['window_size']
word2idx    = cfg["word2idx"]
idx2word    = {i: w for w, i in word2idx.items()}

In [34]:
print("\n" + "="*40)

test_word = "evil"
top_k = 5
print(f"Top {top_k} similar words to '{test_word}':")
res = validate.find_similar_words(
    test_word, 
    word_embeddings_n, 
    word2idx, 
    idx2word,
    k=top_k
)

for w, score in res:
    print(f"  {w:15} {score:.4f}")
print("="*40)


Top 5 similar words to 'evil':
  claiming        0.3658
  serves          0.3514
  straightforward 0.3436
  tel             0.3406
  indispensable   0.3402


In [27]:
word_1 = "he"
word_2 = "she"

v1 = word_embeddings[word2idx[word_1]]
v2 = word_embeddings[word2idx[word_2]]
print(f"Original: {F.cosine_similarity(v1, v2, dim=0).item()}")

Original: 0.7884904146194458


In [28]:
words = [
    "vision",
    "color",
    "red",
    "orange",
    "yellow",
    "green",
    "blue",
    "violet",
    "purple",
    "lilac",
    "taste",
    "bitter",
    "sweet",
    "sour"
]

In [29]:
embedding_size = cfg['embedding_shape'][1]
embeddings = torch.zeros(len(words), embedding_size)
for i in range(len(embeddings)):
    embeddings[i] = word_embeddings[word2idx[words[i]]]

thresholds = [0.2, 0.4, 1, 3, 5, 6, 7]
# thresholds = [i for i in range(1, 11)]

for t in thresholds:
    
    # filtering
    B = (embeddings >= t).int()
    # C = torch.cov(B.T)
    C = B @ B.T

    print(f" -- threshold = {t}, nonzoer(B) = {len((B == 1).nonzero())}")

    df = pd.DataFrame(C, index=words, columns=words)

    df.to_csv(f"model_artifacts/filtration_1-cov_matrix_{t}.csv")

 -- threshold = 0.2, nonzoer(B) = 1546
 -- threshold = 0.4, nonzoer(B) = 1294
 -- threshold = 1, nonzoer(B) = 732
 -- threshold = 3, nonzoer(B) = 47
 -- threshold = 5, nonzoer(B) = 0
 -- threshold = 6, nonzoer(B) = 0
 -- threshold = 7, nonzoer(B) = 0


In [30]:
embeddings = torch.zeros(len(words), embedding_size)
for i in range(len(embeddings)):
    embeddings[i] = word_embeddings[word2idx[words[i]]]

deltas = [3, 2, 1, 0.5, 0.3, 0.1, 0.05]

for d in deltas:
    # filtering
    B = (torch.abs(embeddings) <= d).int()
    C = B @ B.T

    print(f"-- delta = {d}, nonzoer(B) = {len((B == 1).nonzero())}")

    df = pd.DataFrame(C, index=words, columns=words)

    df.to_csv(f"model_artifacts/filtration_2-cov_matrix_{d}.csv")

-- delta = 3, nonzoer(B) = 3496
-- delta = 2, nonzoer(B) = 3199
-- delta = 1, nonzoer(B) = 2176
-- delta = 0.5, nonzoer(B) = 1273
-- delta = 0.3, nonzoer(B) = 811
-- delta = 0.1, nonzoer(B) = 276
-- delta = 0.05, nonzoer(B) = 133
